In [1]:
%reload_ext autoreload
%autoreload 2
import colabexts
from colabexts.jcommon import *

import os, sys, datetime, re, json, importlib
from collections import defaultdict
from sys import modules
from IPython.display import HTML, Javascript
import warnings
import pandas as pd
warnings.filterwarnings('ignore')

%matplotlib inline
pd.set_option('display.max_colwidth', 40)
pd.set_option('display.max_rows', 16)

In [17]:
%%writefile ../geoapp/wakeword.py
#!/usr/bin/env python

# DO NOT EDIT THIS FILE : Generated from scribe/notebooks/wakeword.ipynb

from  mangorest.mango import webapi
import colabexts.utils as colabexts_utils
import logging, os
import numpy as np
logger = logging.getLogger( "geoapp" )

# ------------------------------------------------------------------------------
owwModel  = None
wakeword_models=os.environ.get("WWORDS", "").split()
if ( not wakeword_models):
    try:
        from django.conf import settings
        if hasattr(settings, 'WWORDS'):
            wakeword_models=settings.WWORDS.split()
    except:
        pass
if ( not wakeword_models):
    wakeword_models = "hey_jarvis alexa".split()

logger.info(f"WakeWord models: {wakeword_models}")
def getModel(wakeword_models = wakeword_models ):
    global owwModel
    from openwakeword import Model
    
    if ( owwModel is None):
        
        logger.info(f"Loading models: {wakeword_models}")
        if (wakeword_models):
            owwModel = Model(wakeword_models=wakeword_models, inference_framework="onnx")
        else:
            owwModel = Model(inference_framework="onnx")
            
            
    return owwModel

@webapi("/geoaudio/detectWakeword/")
def detectWakeword(request=None, audio_bytes=None, **kwargs): 
    if ( request and request.FILES.getlist('file')):
        for f in request.FILES.getlist('file'):
            audio_bytes = f.read()
            break;
    if (not audio_bytes):
        return dict(activations=[])
    
    # Add extra bytes of silence if needed
    if len(audio_bytes) % 2 == 1:
        audio_bytes += (b'\x00')

    # Convert audio to correct format and sample rate
    data = np.frombuffer(audio_bytes, dtype=np.int16)
    #if sample_rate != 16000:
    #    data = resampy.resample(data, sample_rate, 16000)

    # Get openWakeWord predictions and set to browser client
    predictions = getModel().predict(data)
    logger.info(predictions)
    ret = []
    for key in predictions:
        if predictions[key] >= 0.3:
            ret.append(key)
            
    if ( not ret):
        ret = ""
    
    return ret

if __name__ == '__main__' and not colabexts_utils.inJupyter():
    detectWakeWord(None, None)

Overwriting ../geoapp/wakeword.py


In [41]:
import openwakeword
from openwakeword.model import Model

# One-time download of all pre-trained models (or only select models)
openwakeword.utils.download_models()

# Instantiate the model(s)
model = Model(
    wakeword_models=["hey jarvis"],  # can also leave this argument empty to load all of the included pre-trained models
)

In [2]:
!ls ../cogs/sage

hey_sage.onnx   hey_sage.tflite sage.onnx       sage.tflite


In [22]:
import openwakeword
from openwakeword.model import Model
import glob
# One-time download of all pre-trained models (or only select models)
openwakeword.utils.download_models()

# Instantiate the model(s)
model = Model(
    # can also leave this argument empty to load all of the included pre-trained models
    wakeword_models=["../cogs/ww/sage.onnx" , "../cogs/ww/hey_sage.onnx"],  
)

import os, numpy as np
files = glob.glob(os.path.expanduser("~/Downloads/sage/examples/*/*.wav") )
ww="sage"
ww1="hey_sage"
for i,f in enumerate(files):
    if ( i > 5):
        break;
    print(i, f)
    preds = model.predict_clip(f)
    
    for p in preds:
        if (p[ww] > 0.5 or p[ww1] > 0.5):
            print(f'Detected: {f} => {p}')
            break


0 /Users/snarayan/Downloads/sage/examples/positive_train/9b2a56f6b96742039eb2a65520d3a719.wav
Detected: /Users/snarayan/Downloads/sage/examples/positive_train/9b2a56f6b96742039eb2a65520d3a719.wav => {'sage': np.float32(0.63085663), 'hey_sage': np.float32(0.00081151724)}
1 /Users/snarayan/Downloads/sage/examples/positive_train/2821d8242f4047daa992ac52dee2ab84.wav
Detected: /Users/snarayan/Downloads/sage/examples/positive_train/2821d8242f4047daa992ac52dee2ab84.wav => {'sage': np.float32(0.9076216), 'hey_sage': np.float32(0.0007728338)}
2 /Users/snarayan/Downloads/sage/examples/positive_train/718c602b203b4ef6907de63e36c81573.wav
Detected: /Users/snarayan/Downloads/sage/examples/positive_train/718c602b203b4ef6907de63e36c81573.wav => {'sage': np.float32(0.8707371), 'hey_sage': np.float32(0.00078877807)}
3 /Users/snarayan/Downloads/sage/examples/positive_train/3a490619e2494a42bb64d5599adca523.wav
Detected: /Users/snarayan/Downloads/sage/examples/positive_train/3a490619e2494a42bb64d5599adca52

In [48]:
ww='hey jarvis'
model = Model(wakeword_models=[ww], inference_framework="onnx")
files=["/tmp/file1.wav", ]
files = glob.glob("/tmp/f*.wav")
for f in files:
    print(f)
    preds = model.predict_clip(f)
    if np.any(p[ww] > 0.5):
        print(f'=> Detected: {f} => {p}')
        
    for p in preds:
        if (p[ww] > 0.5):
            print(f'Detected: {f} => {p}')
            break

In [57]:
%%writefile ../geoapp/oww.py
#!/usr/bin/env python
# 
import sounddevice as sd
import numpy as np
from openwakeword.model import Model

# Load the model with the desired wakeword
oww = Model()
#owm = Model(wakeword_models=["hey jarvis"], inference_framework="onnx")

#oww.enable_custom_wakeword("hey_jarvis")  # built-in wakeword in openWakeWord

# Constants
SAMPLE_RATE = 16000  # must match model's expected rate
BLOCK_SIZE = 10240
CHANNELS = 1

print("Listening for 'Hey Jarvis'...")
result, i, audio_data, paudio_data =0,0, [], []
def audio_callback(indata, frames, time, status):
    if status:
        print(f"Error: {status}")
        return
    global result, i, audio_data, paudio_data
    
    i += 1
    # Preprocess audio
    audio_data = np.squeeze(indata)
    #audio_data = indata
    if ( np.all(audio_data <0.09) ):
        print(f"SILENT: {len(paudio_data)} { len(audio_data)} {audio_data[0]} \r", end="")
        paudio_data = audio_data
        return
    elif ( len(paudio_data) > 0 and np.all(audio_data == paudio_data) ):
        print(f"DIFF/SAME: {len(paudio_data)} { len(audio_data)} {paudio_data}")
        #return
    sd.play(indata,  blocking=True)
        
    paudio_data = audio_data
    print()
    # Feed audio to the model
    result = oww.predict(audio_data)
    print(f'{i} Result: {result["hey_jarvis"]} {result["alexa"]} {time} Status: {status}\r', end="\n")

    # Check for trigger
    #if result["hey_jarvis"] > 0.5:
    #    print("==> Wake word 'Hey Jarvis' detected!")
    for t in result:
        if result[t] > 0.5:
            print( f"==> {t} Wake word detected!")
            

# Start audio stream
with sd.InputStream(callback=audio_callback,
                    channels=CHANNELS,
                    samplerate=SAMPLE_RATE,
                    blocksize=BLOCK_SIZE):
    input("Press Enter to stop...\n")


Overwriting ../geoapp/oww.py


In [16]:
result

{'alexa': 1.1026859e-06,
 'hey_mycroft': -5.9604645e-08,
 'hey_jarvis': 5.096197e-06,
 'hey_rhasspy': 0.0017027259,
 '1_minute_timer': 1.6118423e-07,
 '5_minute_timer': 1.6118423e-07,
 '10_minute_timer': 6.6065303e-07,
 '20_minute_timer': 1.6118423e-07,
 '30_minute_timer': 1.6841341e-07,
 '1_hour_timer': 1.6118423e-07,
 'weather': 1.5795231e-06}